# Capítulo 7 — La Integral Definida y sus Aplicaciones
## Métodos Cuantitativos para la Economía
**Autor:** MSc. Jeel Cueva | Lima, Perú, 2025

---
### Contenido
1. Sumas de Riemann — Convergencia  
2. Teorema Fundamental del Cálculo (TFC)  
3. Cálculo de Áreas entre Curvas  
4. Excedente del Consumidor y del Productor  
5. Valor Presente de Flujos Continuos  
6. Integrales Impropias y Horizonte Infinito  
7. Teorema del Valor Medio para Integrales  


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CONFIGURACION GLOBAL — Capitulo 7
# ═══════════════════════════════════════════════════════════════
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.integrate import quad
from scipy.optimize import fsolve, brentq
import warnings
warnings.filterwarnings('ignore')

C_AZUL='#1f77b4'; C_ROJO='#d62728'; C_VERDE='#2ca02c'
C_NARANJA='#ff7f0e'; C_MORADO='#9467bd'; C_CAFE='#8c564b'

plt.rcParams.update({
    'figure.dpi':110,'axes.grid':True,'grid.alpha':0.3,
    'axes.spines.top':False,'axes.spines.right':False,
    'font.size':11,'axes.titlesize':13,'axes.labelsize':11
})
print("Capitulo 7 — La Integral Definida y sus Aplicaciones")
print("Librerias listas")


---
## Sección 7.1 — Sumas de Riemann: Convergencia al Área Exacta
### Ejemplo 7.1 — f(x) = x² en [0,1]: puntos izquierdo, derecho y medio

$$S_n = \sum_{i=1}^n f(c_i)\,\Delta x \xrightarrow{n\to\infty} \int_0^1 x^2\,dx = \frac{1}{3}$$


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Ejemplo 7.1 — Sumas de Riemann: Convergencia
# ═══════════════════════════════════════════════════════════════
def suma_riemann(f, a, b, n, metodo='derecha'):
    dx = (b - a) / n
    x_pts = np.linspace(a, b, n+1)
    if metodo == 'izquierda':
        c = x_pts[:-1]
    elif metodo == 'derecha':
        c = x_pts[1:]
    else:  # punto medio
        c = (x_pts[:-1] + x_pts[1:]) / 2
    return np.sum(f(c)*dx), c, dx

f = lambda x: x**2
a, b = 0, 1
valor_exacto = 1/3
print(f"─── Sumas de Riemann para f(x)=x² en [0,1] ───")
print(f"  Valor exacto: ∫₀¹ x² dx = 1/3 ≈ {valor_exacto:.8f}")
print()
print(f"  {'n':>6} {'S_izq':>12} {'S_der':>12} {'S_med':>12} {'Error_der':>12}")
print("  " + "─"*60)
for n in [4, 10, 50, 100, 1000]:
    s_izq = suma_riemann(f, a, b, n, 'izquierda')[0]
    s_der = suma_riemann(f, a, b, n, 'derecha')[0]
    s_med = suma_riemann(f, a, b, n, 'medio')[0]
    err = abs(s_der - valor_exacto)
    print(f"  {n:>6}  {s_izq:>12.6f}  {s_der:>12.6f}  {s_med:>12.6f}  {err:>12.2e}")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Ejemplo 7.1 — Convergencia de Sumas de Riemann para f(x)=x²', fontsize=13, fontweight='bold')
x_plot = np.linspace(0, 1, 500)
f_plot = x_plot**2

for ax, n, titulo in zip(axes, [4, 10, 50], ['n=4 (aprox. gruesa)', 'n=10', 'n=50 (fina)']):
    s_der, c_pts, dx = suma_riemann(f, a, b, n, 'derecha')
    s_izq, c_izq, _  = suma_riemann(f, a, b, n, 'izquierda')
    # Dibujar rectángulos derechos
    for ci, dxi in zip(c_pts, [dx]*n):
        ax.bar(ci-dxi, f(ci), width=dxi, align='edge', alpha=0.4,
               color=C_AZUL, edgecolor='navy', linewidth=0.5)
    ax.plot(x_plot, f_plot, color=C_ROJO, lw=2.5, label='f(x)=x²', zorder=5)
    ax.set_xlabel('x'); ax.set_ylabel('f(x)')
    ax.set_title(f'{titulo}
S_der={s_der:.4f}, exacto={valor_exacto:.4f}')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.1)
    ax.text(0.05, 0.90, f'Error={abs(s_der-valor_exacto):.4f}',
            transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle='round', facecolor='lightyellow'))
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('cap07_fig01_riemann.png', bbox_inches='tight', dpi=110)
plt.show()
print("Figura guardada: cap07_fig01_riemann.png")


---
### Ejemplo 7.2 — Comparación: Extremos Izquierdos, Derechos y Puntos Medios

Función: $f(x)=2x+1$ en $[0,2]$ con $n=4$.  
Valor exacto: $\int_0^2(2x+1)dx=[x^2+x]_0^2 = 6$.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Ejemplo 7.2 — Comparacion de Tipos de Puntos (n=4)
# ═══════════════════════════════════════════════════════════════
f = lambda x: 2*x + 1
a, b, n = 0, 2, 4
valor_exacto = 6.0

s_izq, c_izq, dx = suma_riemann(f, a, b, n, 'izquierda')
s_der, c_der, _   = suma_riemann(f, a, b, n, 'derecha')
s_med, c_med, _   = suma_riemann(f, a, b, n, 'medio')

print(f"─── f(x)=2x+1 en [0,2], n={n} ───")
print(f"  Valor exacto = {valor_exacto}")
print(f"  S_izquierdo  = {s_izq:.4f}  (error = {s_izq-valor_exacto:+.4f})")
print(f"  S_derecho    = {s_der:.4f}  (error = {s_der-valor_exacto:+.4f})")
print(f"  S_medio      = {s_med:.4f}  (error = {s_med-valor_exacto:+.4f})")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Ejemplo 7.2 — Comparacion Izquierdo / Derecho / Medio (n=4)', fontsize=13, fontweight='bold')
x_plot = np.linspace(0, 2, 300)
colores_tipos = [('izquierda', C_AZUL, c_izq, s_izq, 'Extremos Izquierdos'),
                 ('derecha',   C_ROJO, c_der, s_der, 'Extremos Derechos'),
                 ('medio',     C_VERDE, c_med, s_med, 'Puntos Medios')]

x_edges = np.linspace(a, b, n+1)
for ax, (metodo, col, c_pts, s_val, titulo) in zip(axes, colores_tipos):
    for i, ci in enumerate(c_pts):
        xi_left = x_edges[i]
        ax.bar(xi_left, f(ci), width=dx, align='edge', alpha=0.45,
               color=col, edgecolor='black', linewidth=0.8)
        ax.plot([xi_left, xi_left+dx], [f(ci), f(ci)], color='black', lw=1.5)
    ax.plot(x_plot, f(x_plot), color='black', lw=2.5, zorder=5, label='f(x)=2x+1')
    ax.set_xlim(-0.1, 2.1); ax.set_ylim(0, 6)
    ax.set_xlabel('x'); ax.set_ylabel('f(x)')
    ax.set_title(f'{titulo}
S={s_val:.2f}, exacto={valor_exacto}, error={s_val-valor_exacto:+.2f}')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('cap07_fig02_comparacion_riemann.png', bbox_inches='tight', dpi=110)
plt.show()
print("Figura guardada: cap07_fig02_comparacion_riemann.png")


---
## Sección 7.2 — Teorema Fundamental del Cálculo
### Ejemplo 7.3 — TFC Parte I y II: Cálculo de Integrales Definidas

**TFC Parte II:** Si $F'=f$, entonces $\int_a^b f(x)\,dx = F(b)-F(a)$.

Casos:  
- (a) $\int_1^4(3x^2+1/x)\,dx$  
- (b) Distancia total: partícula con $v(t)=3t^2-12t+9$  
- (c) $G(x)=\int_0^x CM(t)\,dt$ — costo acumulado  


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Ejemplo 7.3 — Teorema Fundamental del Calculo
# ═══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Ejemplo 7.3 — Teorema Fundamental del Calculo', fontsize=14, fontweight='bold')

# ── (a) ∫₁⁴ (3x² + 1/x)dx ───────────────────────────────────
ax = axes[0]
f_a = lambda x: 3*x**2 + 1/x
F_a = lambda x: x**3 + np.log(x)
a_a, b_a = 1, 4
val_a = F_a(b_a) - F_a(a_a)
xa = np.linspace(1, 4, 400)
ax.plot(xa, f_a(xa), color=C_AZUL, lw=2.5, label='f(x)=3x²+1/x')
ax.fill_between(xa, 0, f_a(xa), alpha=0.25, color=C_AZUL,
                label=f'Area = {val_a:.4f}')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title(f'(a) ∫₁⁴(3x²+1/x)dx = {val_a:.2f}')
ax.legend(fontsize=9)
val_a_num, _ = quad(f_a, 1, 4)
print(f"─── (a): ∫₁⁴(3x²+1/x)dx ───")
print(f"  Por TFC: [x³+ln x]₁⁴ = ({4**3}+ln 4) - (1+0)")
print(f"         = ({4**3}+{np.log(4):.4f}) - 1 = {val_a:.4f}")
print(f"  Verificacion numerica: {val_a_num:.4f}  ✓")

# ── (b) Distancia total: v(t)=3t²-12t+9 ────────────────────
ax = axes[1]
v = lambda t: 3*t**2 - 12*t + 9
t_plot = np.linspace(0, 4, 400)
ax.plot(t_plot, v(t_plot), color=C_ROJO, lw=2.5, label='v(t)=3t²-12t+9')
ax.axhline(0, color='black', lw=1)
# Raices: v=0 → 3(t-1)(t-3)=0 → t=1, t=3
for ti, lbl in [(1, 't=1'), (3, 't=3')]:
    ax.axvline(ti, color='gray', lw=1, ls='--')
    ax.text(ti+0.05, 1, lbl, fontsize=9, color='gray')
# Zonas
t1, t2 = np.linspace(0,1,100), np.linspace(1,3,100)
t3 = np.linspace(3,4,100)
ax.fill_between(t1, 0, v(t1), alpha=0.3, color=C_VERDE, label='Fwd: A₁=4')
ax.fill_between(t2, 0, v(t2), alpha=0.3, color=C_ROJO,  label='Back: A₂=4')
ax.fill_between(t3, 0, v(t3), alpha=0.3, color=C_NARANJA,label='Fwd: A₃=4')
ax.set_xlabel('t (seg)'); ax.set_ylabel('v(t) [m/s]')
ax.set_title('(b) Distancia Total en [0,4]')
ax.legend(fontsize=8)

s = lambda t: t**3 - 6*t**2 + 9*t  # antiderivada
A1 = s(1)-s(0)
A2 = abs(s(3)-s(1))
A3 = s(4)-s(3)
print(f"
─── (b): Distancia Total ───")
print(f"  v(t)=3t²-12t+9=3(t-1)(t-3), raices t=1,3")
print(f"  A₁ = ∫₀¹ v dt  = {A1:.1f} m (adelante)")
print(f"  A₂ = |∫₁³ v dt| = {A2:.1f} m (atras)")
print(f"  A₃ = ∫₃⁴ v dt  = {A3:.1f} m (adelante)")
print(f"  Distancia TOTAL  = {A1+A2+A3:.1f} m")
print(f"  Desplazamiento   = ∫₀⁴ v dt = {s(4)-s(0):.1f} m")

# ── (c) Costo acumulado G(x) = ∫₀ˣ CM(t)dt ─────────────────
ax = axes[2]
CM = lambda t: 50 + 20*t + 0.3*t**2
x_arr = np.linspace(0, 15, 400)
G_x = np.array([quad(CM, 0, xi)[0] for xi in x_arr])
ax.plot(x_arr, G_x, color=C_VERDE, lw=2.5, label='G(x)=∫₀ˣ CM(t)dt')
ax.plot(x_arr, CM(x_arr)*10, color=C_ROJO, lw=2, ls='--', label='CM(x)×10')
x_ref = 10
G_ref = quad(CM, 0, x_ref)[0]
ax.plot(x_ref, G_ref, 'r*', ms=12, label=f'G(10)={G_ref:.0f}')
ax.set_xlabel('x (miles unidades)'); ax.set_ylabel('Miles S/')
ax.set_title('(c) Costo Acumulado G(x)=∫₀ˣ CM dt
G'(x)=CM(x) por TFC Parte I')
ax.legend(fontsize=9)
print(f"
─── (c): Costo Acumulado G(x) ───")
print(f"  CM(x)=50+20x+0.3x², G(x)=∫₀ˣ CM dt")
print(f"  G'(x) = CM(x)  (TFC Parte I)")
print(f"  G(10) = {G_ref:.2f} miles de S/")
print(f"  G'(10)= CM(10)= {CM(10):.2f} miles S/ por mil unidades")

plt.tight_layout()
plt.savefig('cap07_fig03_tfc.png', bbox_inches='tight', dpi=110)
plt.show()
print("
Figura guardada: cap07_fig03_tfc.png")


---
## Sección 7.3 — Cálculo de Áreas entre Curvas
### Ejemplo 7.4 — Áreas: Parabola-Recta, Tres Intersecciones, Respecto a y

$$A = \int_a^b [f(x) - g(x)]\,dx \quad \text{si } f(x) \geq g(x)$$


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Ejemplo 7.4 — Areas entre Curvas
# ═══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Ejemplo 7.4 — Areas entre Curvas', fontsize=14, fontweight='bold')

# ── (a) f(x)=x²+1 y g(x)=2x+4 ──────────────────────────────
ax = axes[0]
f_a = lambda x: x**2 + 1
g_a = lambda x: 2*x + 4
# Intersecciones: x²-2x-3=0 → (x-3)(x+1)=0 → x=-1,3
xa = np.linspace(-2, 4, 400)
ax.plot(xa, f_a(xa), color=C_AZUL, lw=2.5, label='f(x)=x²+1')
ax.plot(xa, g_a(xa), color=C_ROJO, lw=2.5, label='g(x)=2x+4')
x_int = np.linspace(-1, 3, 300)
ax.fill_between(x_int, f_a(x_int), g_a(x_int), alpha=0.3, color=C_VERDE,
                label='Area entre curvas')
A_a = quad(lambda x: g_a(x)-f_a(x), -1, 3)[0]
ax.plot([-1,3], [f_a(-1), f_a(3)], 'ko', ms=8)
ax.set_ylim(-1, 14); ax.set_xlabel('x')
ax.set_title(f'(a) f=x²+1, g=2x+4
Area={A_a:.4f}')
ax.legend(fontsize=9)
print(f"─── (a): Area entre parabolay recta ───")
print(f"  Intersecciones: x=-1, x=3")
print(f"  A = ∫₋₁³ [(2x+4)-(x²+1)] dx = {A_a:.4f}  ≈ 32/3={32/3:.4f}  ✓")

# ── (b) f(x)=x³-3x y g(x)=x ────────────────────────────────
ax = axes[1]
f_b = lambda x: x**3 - 3*x
g_b = lambda x: x
# Intersecciones: x³-4x=0 → x=0,±2
xb = np.linspace(-2.5, 2.5, 400)
ax.plot(xb, f_b(xb), color=C_AZUL, lw=2.5, label='f(x)=x³-3x')
ax.plot(xb, g_b(xb), color=C_ROJO, lw=2.5, label='g(x)=x')
x_izq = np.linspace(-2, 0, 200)
x_der = np.linspace(0, 2, 200)
ax.fill_between(x_izq, f_b(x_izq), g_b(x_izq),
                where=f_b(x_izq)>=g_b(x_izq), alpha=0.3, color=C_VERDE, label='A₁=4')
ax.fill_between(x_der, g_b(x_der), f_b(x_der),
                where=g_b(x_der)>=f_b(x_der), alpha=0.3, color=C_NARANJA, label='A₂=4')
A_b1 = quad(lambda x: f_b(x)-g_b(x), -2, 0)[0]
A_b2 = quad(lambda x: g_b(x)-f_b(x), 0, 2)[0]
ax.set_xlabel('x')
ax.set_title(f'(b) Tres intersecciones
A_total={A_b1+A_b2:.1f}')
ax.legend(fontsize=9)
ax.axhline(0, color='black', lw=0.5)
print(f"
─── (b): Tres intersecciones ───")
print(f"  f=x³-3x, g=x: intersecciones en x=-2,0,2")
print(f"  A₁=∫₋₂⁰ (f-g) dx = {A_b1:.4f}")
print(f"  A₂=∫₀² (g-f) dx  = {A_b2:.4f}")
print(f"  A_total = {A_b1+A_b2:.4f}")

# ── (c) Respecto a y: x=y² y x=y+2 ─────────────────────────
ax = axes[2]
y_rng = np.linspace(-1.5, 2.5, 400)
x1 = y_rng**2
x2 = y_rng + 2
ax.plot(x1, y_rng, color=C_AZUL, lw=2.5, label='x=y²')
ax.plot(x2, y_rng, color=C_ROJO, lw=2.5, label='x=y+2')
y_int = np.linspace(-1, 2, 300)
ax.fill_betweenx(y_int, y_int**2, y_int+2, alpha=0.3, color=C_MORADO,
                 label='Area respecto a y')
ax.plot([(-1)**2, 2**2], [-1, 2], 'ko', ms=8)
A_c = quad(lambda y: (y+2)-y**2, -1, 2)[0]
ax.set_xlim(-0.5, 6); ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title(f'(c) Integracion respecto a y
A={A_c:.4f}=9/2={9/2:.4f}')
ax.legend(fontsize=9)
print(f"
─── (c): Respecto a y ───")
print(f"  x=y², x=y+2: intersecciones en y=-1,2")
print(f"  A=∫₋₁² [(y+2)-y²] dy = {A_c:.4f}  ≈ 9/2={9/2:.4f}  ✓")

plt.tight_layout()
plt.savefig('cap07_fig04_areas.png', bbox_inches='tight', dpi=110)
plt.show()
print("
Figura guardada: cap07_fig04_areas.png")


---
## Sección 7.4 — Excedente del Consumidor y del Productor
### Ejemplo 7.5 — Mercado Lineal: EC, EP y Pérdida por Impuesto

$$EC = \int_0^{Q^*}[D(Q)-P^*]\,dQ \qquad EP = \int_0^{Q^*}[P^*-O(Q)]\,dQ$$


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Ejemplo 7.5 — Excedentes del Consumidor y Productor
# ═══════════════════════════════════════════════════════════════
# ── Mercado de referencia ────────────────────────────────────
D = lambda Q: 200 - 2*Q   # Demanda
O = lambda Q: 20 + Q      # Oferta
# Equilibrio: 200-2Q=20+Q → Q*=60, P*=80
Q_eq = 60; P_eq = 80

EC = quad(lambda Q: D(Q) - P_eq, 0, Q_eq)[0]
EP = quad(lambda Q: P_eq - O(Q), 0, Q_eq)[0]

print(f"─── Mercado Lineal: D={200}-2Q, O=20+Q ───")
print(f"  Equilibrio: Q*={Q_eq}, P*={P_eq}")
print(f"  EC = ∫₀⁶⁰ [{200}-2Q-{P_eq}] dQ = {EC:.2f}")
print(f"  EP = ∫₀⁶⁰ [{P_eq}-(20+Q)] dQ  = {EP:.2f}")
print(f"  Bienestar Total = EC+EP = {EC+EP:.2f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Ejemplo 7.5 — Excedentes: Consumidor, Productor e Impuesto', fontsize=13, fontweight='bold')

Q_plot = np.linspace(0, 90, 400)

# ── Panel a: EC y EP ────────────────────────────────────────
ax = axes[0]
ax.plot(Q_plot, D(Q_plot), color=C_AZUL, lw=2.5, label='D(Q)=200-2Q')
ax.plot(Q_plot, O(Q_plot), color=C_ROJO, lw=2.5, label='O(Q)=20+Q')
Q_fill = np.linspace(0, Q_eq, 300)
ax.fill_between(Q_fill, P_eq, D(Q_fill), alpha=0.35, color=C_AZUL, label=f'EC={EC:.0f}')
ax.fill_between(Q_fill, O(Q_fill), P_eq, alpha=0.35, color=C_ROJO, label=f'EP={EP:.0f}')
ax.axhline(P_eq, color='gray', lw=1.5, ls='--', label=f'P*={P_eq}')
ax.axvline(Q_eq, color='gray', lw=1.5, ls='--', label=f'Q*={Q_eq}')
ax.plot(Q_eq, P_eq, 'k*', ms=12)
ax.set_xlabel('Cantidad Q'); ax.set_ylabel('Precio P')
ax.set_title(f'EC={EC:.0f}, EP={EP:.0f}
Bienestar={EC+EP:.0f}')
ax.legend(fontsize=8, loc='upper right')

# ── Panel b: Impuesto t=10 ───────────────────────────────────
ax = axes[1]
t_imp = 10  # impuesto por unidad
# Nueva oferta: O_t(Q) = 20 + Q + 10 = 30 + Q
O_t = lambda Q: O(Q) + t_imp
# Nuevo equilibrio: 200-2Q = 30+Q → Q_t=170/3≈56.67
Q_t = (200-30)/3; P_c = D(Q_t); P_p = O(Q_t)  # precio consumidor y productor
EC_t = quad(lambda Q: D(Q)-P_c, 0, Q_t)[0]
EP_t = quad(lambda Q: P_p-O(Q), 0, Q_t)[0]
Rec  = t_imp*Q_t
PIE  = EC+EP - EC_t - EP_t - Rec

ax.plot(Q_plot, D(Q_plot), color=C_AZUL, lw=2.5, label='D(Q)')
ax.plot(Q_plot, O(Q_plot), color=C_ROJO,  lw=2.5, label='O original')
ax.plot(Q_plot, O_t(Q_plot), color=C_NARANJA, lw=2.5, ls='--', label=f'O+t (t={t_imp})')
Q_fill_t = np.linspace(0, Q_t, 300)
ax.fill_between(Q_fill_t, P_c, D(Q_fill_t), alpha=0.3, color=C_AZUL, label=f'EC={EC_t:.1f}')
ax.fill_between(Q_fill_t, O(Q_fill_t), P_p, alpha=0.3, color=C_ROJO, label=f'EP={EP_t:.1f}')
ax.fill_between(Q_fill_t, P_p, P_c, alpha=0.3, color=C_VERDE, label=f'Recaudacion={Rec:.1f}')
ax.axhline(P_c, color=C_AZUL, lw=1, ls=':', alpha=0.7, label=f'P_c={P_c:.1f}')
ax.axhline(P_p, color=C_ROJO, lw=1, ls=':', alpha=0.7, label=f'P_p={P_p:.1f}')
ax.set_xlabel('Cantidad Q'); ax.set_ylabel('Precio P')
ax.set_title(f'Impuesto t={t_imp}
PIE={PIE:.2f}')
ax.legend(fontsize=7, loc='upper right')

print(f"
─── Con Impuesto t={t_imp} ───")
print(f"  Nuevo Q*={Q_t:.2f}, P_consumidor={P_c:.2f}, P_productor={P_p:.2f}")
print(f"  EC={EC_t:.2f}, EP={EP_t:.2f}, Recaudacion={Rec:.2f}")
print(f"  Perdida Irrecuperable = {PIE:.2f}")

# ── Panel c: Precio Maximo ───────────────────────────────────
ax = axes[2]
P_max = 60  # precio máximo por debajo del equilibrio (80)
Q_d_max = (200-P_max)/2  # cantidad demandada al precio max
Q_o_max = P_max - 20     # cantidad ofrecida al precio max
Q_transada = min(Q_d_max, Q_o_max)

ax.plot(Q_plot, D(Q_plot), color=C_AZUL, lw=2.5, label='D(Q)=200-2Q')
ax.plot(Q_plot, O(Q_plot), color=C_ROJO, lw=2.5, label='O(Q)=20+Q')
Q_fill_m = np.linspace(0, Q_transada, 300)
EC_max = quad(lambda Q: D(Q)-P_max, 0, Q_transada)[0]
EP_max = quad(lambda Q: P_max-O(Q), 0, Q_transada)[0]
ax.fill_between(Q_fill_m, P_max, D(Q_fill_m), alpha=0.35, color=C_AZUL, label=f'EC={EC_max:.1f}')
ax.fill_between(Q_fill_m, O(Q_fill_m), P_max, alpha=0.35, color=C_ROJO, label=f'EP={EP_max:.1f}')
ax.axhline(P_max, color=C_VERDE, lw=2, ls='--', label=f'P_max={P_max}')
ax.axvline(Q_transada, color='gray', lw=1, ls=':')
ax.annotate(f'Escasez={Q_d_max-Q_transada:.0f}', xy=(Q_transada, P_max),
            xytext=(Q_transada+8, P_max+15),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=9)
ax.set_xlabel('Cantidad Q'); ax.set_ylabel('Precio P')
ax.set_title(f'Precio Max={P_max}: Escasez={Q_d_max-Q_transada:.0f}')
ax.legend(fontsize=8, loc='upper right')

print(f"
─── Precio Maximo P={P_max} ───")
print(f"  Q_demandada={Q_d_max:.1f}, Q_ofrecida={Q_transada:.1f}")
print(f"  Escasez = {Q_d_max-Q_transada:.1f} unidades")
print(f"  EC(con precio max) = {EC_max:.2f}  (vs EC sin={EC:.2f})")
print(f"  EP(con precio max) = {EP_max:.2f}  (vs EP sin={EP:.2f})")

plt.tight_layout()
plt.savefig('cap07_fig05_excedentes.png', bbox_inches='tight', dpi=110)
plt.show()
print("
Figura guardada: cap07_fig05_excedentes.png")


---
## Sección 7.5 — Valor Presente de Flujos Continuos
### Ejemplo 7.6 — Proyecto con flujo creciente y comparación de proyectos

$$VP = \int_0^T R(t)\,e^{-rt}\,dt$$


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Ejemplo 7.6 — Valor Presente de Flujos Continuos
# ═══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Ejemplo 7.6 — Valor Presente de Flujos Continuos', fontsize=14, fontweight='bold')

# ── (a) Flujo creciente: R(t)=50000+2000t, r=0.08, T=10 ─────
ax = axes[0]
R_a = lambda t: 50000 + 2000*t
r_a = 0.08; T_a = 10
t_a = np.linspace(0, T_a, 400)
flujo_nom = R_a(t_a)
flujo_des = R_a(t_a)*np.exp(-r_a*t_a)
VP_a = quad(lambda t: R_a(t)*np.exp(-r_a*t), 0, T_a)[0]
ax.plot(t_a, flujo_nom/1000, color=C_AZUL, lw=2.5, label='R(t) nominal')
ax.plot(t_a, flujo_des/1000, color=C_ROJO, lw=2, ls='--', label='R(t)·e^{-rt} descontado')
ax.fill_between(t_a, 0, flujo_des/1000, alpha=0.25, color=C_ROJO, label=f'VP=S/{VP_a/1000:.1f}k')
ax.set_xlabel('Años'); ax.set_ylabel('Miles S/año')
ax.set_title(f'(a) R(t)=50k+2kt
VP={VP_a/1000:.1f}k, r={r_a:.0%}')
ax.legend(fontsize=8)
I0_a = 350000
VPN_a = VP_a - I0_a
print(f"─── (a): Flujo creciente ───")
print(f"  VP = {VP_a:,.2f}  |  I₀ = {I0_a:,}  |  VPN = {VPN_a:,.2f}")
print(f"  {'Rentable' if VPN_a>0 else 'No rentable'}")

# ── (b) Comparacion proyectos A y B ─────────────────────────
ax = axes[1]
r_b = 0.10; T_b = 12
R_A = lambda t: 25000 + 1000*t   # Proyecto A
R_B = lambda t: 22000 + 1500*t   # Proyecto B
I0_A, I0_B = 200000, 180000

t_b = np.linspace(0, T_b, 400)
VP_A = quad(lambda t: R_A(t)*np.exp(-r_b*t), 0, T_b)[0]
VP_B = quad(lambda t: R_B(t)*np.exp(-r_b*t), 0, T_b)[0]
VPN_A = VP_A - I0_A; VPN_B = VP_B - I0_B

ax.plot(t_b, (R_A(t_b)*np.exp(-r_b*t_b))/1000, color=C_AZUL, lw=2.5, label=f'A: VP={VP_A/1000:.1f}k')
ax.plot(t_b, (R_B(t_b)*np.exp(-r_b*t_b))/1000, color=C_ROJO, lw=2.5, ls='--', label=f'B: VP={VP_B/1000:.1f}k')
ax.fill_between(t_b, 0, (R_A(t_b)*np.exp(-r_b*t_b))/1000, alpha=0.15, color=C_AZUL)
ax.fill_between(t_b, 0, (R_B(t_b)*np.exp(-r_b*t_b))/1000, alpha=0.15, color=C_ROJO)
ax.set_xlabel('Años'); ax.set_ylabel('Flujo descontado (miles S/)')
ax.set_title(f'(b) Comparacion Proyectos
VPN_A={VPN_A/1000:.1f}k  VPN_B={VPN_B/1000:.1f}k')
ax.legend(fontsize=9)
print(f"
─── (b): Comparacion de Proyectos ───")
for nombre, vp, vpn, i0 in [('A', VP_A, VPN_A, I0_A), ('B', VP_B, VPN_B, I0_B)]:
    print(f"  Proyecto {nombre}: VP={vp:,.0f}  I₀={i0:,}  VPN={vpn:,.0f}")
mejor = 'A' if VPN_A > VPN_B else 'B'
print(f"  Mejor proyecto: {mejor}")

# ── (c) Sensibilidad VP a la tasa ───────────────────────────
ax = axes[2]
tasas = np.linspace(0.01, 0.25, 400)
R_c = 15000  # flujo constante
T_c = 8

VP_sens = np.array([quad(lambda t: R_c*np.exp(-r*t), 0, T_c)[0] for r in tasas])
ax.plot(tasas*100, VP_sens/1000, color=C_VERDE, lw=2.5, label='VP(r)')
ax.axhline(80, color=C_ROJO, lw=1.5, ls='--', label='I₀=80k')
# TIR
tir = fsolve(lambda r: quad(lambda t: R_c*np.exp(-r[0]*t), 0, T_c)[0]-80000, 0.1)[0]
ax.axvline(tir*100, color=C_NARANJA, lw=1.5, ls='-.', label=f'TIR={tir:.1%}')
ax.set_xlabel('Tasa de Descuento (%)'); ax.set_ylabel('Valor Presente (miles S/)')
ax.set_title(f'(c) Sensibilidad VP
R={R_c/1000:.0f}k/año, T={T_c} años')
ax.legend(fontsize=9)
print(f"
─── (c): Sensibilidad VP ───")
print(f"  TIR del proyecto: {tir:.2%}")
for r_val in [0.05, 0.08, 0.10, 0.12, tir]:
    vp_val = quad(lambda t: R_c*np.exp(-r_val*t), 0, T_c)[0]
    print(f"  r={r_val:.0%}: VP={vp_val:,.0f}")

plt.tight_layout()
plt.savefig('cap07_fig06_valor_presente.png', bbox_inches='tight', dpi=110)
plt.show()
print("
Figura guardada: cap07_fig06_valor_presente.png")


---
## Sección 7.6 — Integrales Impropias y Horizonte Infinito
### Ejemplo 7.7 — p-Integral, Perpetuidades y Distribución de Pareto

**Criterio p-Integral:**  
$$\int_1^{+\infty}\frac{dx}{x^p} = \begin{cases} \frac{1}{p-1} & p>1 \\ +\infty & p\leq 1 \end{cases}$$


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Ejemplo 7.7 — Integrales Impropias
# ═══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Ejemplo 7.7 — Integrales Impropias', fontsize=14, fontweight='bold')

# ── (a) p-Integral: ∫₁^∞ x^{-p} dx ─────────────────────────
ax = axes[0]
x_plot = np.linspace(1, 8, 400)
p_vals = [0.5, 1.0, 1.5, 2.0, 3.0]
colores_p = [C_ROJO, C_NARANJA, C_VERDE, C_AZUL, C_MORADO]
for p, col in zip(p_vals, colores_p):
    y = x_plot**(-p)
    if p > 1:
        val = 1/(p-1)
        ax.plot(x_plot, y, color=col, lw=2.5, label=f'p={p} → ∫={val:.2f} (CONV)')
    else:
        ax.plot(x_plot, y, color=col, lw=2.5, ls='--', label=f'p={p} → DIV')
ax.fill_between(x_plot, 0, x_plot**(-2), alpha=0.15, color=C_AZUL)
ax.set_ylim(0, 2); ax.set_xlabel('x'); ax.set_ylabel('x^{-p}')
ax.set_title('(a) p-Integral: ∫₁^∞ x^{-p} dx')
ax.legend(fontsize=8)
print("─── p-Integral ───")
print(f"  {'p':>5} {'Resultado':>15} {'Convergencia':>15}")
for p in [0.5, 1.0, 1.5, 2.0, 3.0]:
    if p > 1:
        val = f"1/(p-1)={1/(p-1):.4f}"
        conv = "CONVERGE"
    else:
        val = "+infinito"
        conv = "DIVERGE"
    print(f"  {p:>5}  {val:>15}  {conv:>15}")

# ── (b) Perpetuidades ────────────────────────────────────────
ax = axes[1]
t_plot = np.linspace(0, 30, 500)
D_val = 5000  # dividendo anual
tasas_perp = [0.04, 0.06, 0.08, 0.10, 0.12]
for r_p, col in zip(tasas_perp, colores_p):
    flujo = D_val*np.exp(-r_p*t_plot)
    VP_perp = D_val/r_p  # limite
    ax.plot(t_plot, flujo, color=col, lw=2, label=f'r={r_p:.0%}, VP=S/{VP_perp:,.0f}')
ax.set_xlabel('Años'); ax.set_ylabel('Flujo Descontado D·e^{-rt}')
ax.set_title('(b) Perpetuidades: VP=D/r
D=S/5000/año')
ax.legend(fontsize=8)
print("
─── Perpetuidades (D=S/5000) ───")
for r_p in tasas_perp:
    vp = D_val/r_p
    print(f"  r={r_p:.0%}: VP = {D_val}/{r_p} = S/{vp:,.0f}")

# ── (c) Distribucion de Pareto ───────────────────────────────
ax = axes[2]
x_par = np.linspace(1, 8, 400)
alpha_vals = [1.5, 2.0, 2.5, 3.0]
x_m = 1  # minimo

for alpha, col in zip(alpha_vals, colores_p[:4]):
    f_pareto = alpha * x_m**alpha / x_par**(alpha+1)
    E_X = alpha*x_m/(alpha-1) if alpha > 1 else float('inf')
    ax.plot(x_par, f_pareto, color=col, lw=2.5, label=f'α={alpha}, E[X]={E_X:.2f}')

ax.set_xlabel('x (ingreso)'); ax.set_ylabel('f(x)')
ax.set_title('(c) Distribucion de Pareto
f(x)=αx_m^α/x^{α+1}')
ax.legend(fontsize=9)
print("
─── Distribucion de Pareto (x_m=1) ───")
for alpha in alpha_vals:
    E_X = alpha/(alpha-1)
    Var = alpha/((alpha-1)**2*(alpha-2)) if alpha > 2 else float('inf')
    print(f"  α={alpha}: E[X]={E_X:.3f}, Var={'inf' if Var==float('inf') else f'{Var:.3f}'}")

# Verificacion: ∫₁^∞ f(x)dx = 1
for alpha in alpha_vals:
    integral, _ = quad(lambda x: alpha*x_m**alpha/x**(alpha+1), 1, np.inf)
    print(f"  α={alpha}: ∫f(x)dx = {integral:.6f}  ✓")

plt.tight_layout()
plt.savefig('cap07_fig07_impropias.png', bbox_inches='tight', dpi=110)
plt.show()
print("
Figura guardada: cap07_fig07_impropias.png")


---
## Sección 7.7 — Teorema del Valor Medio para Integrales
### Ejemplo 7.8 — Costo Medio de Producción y Aplicaciones

$$\bar{f} = \frac{1}{b-a}\int_a^b f(x)\,dx, \quad \exists c \in (a,b): f(c)=\bar{f}$$


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Ejemplo 7.8 — Teorema del Valor Medio para Integrales
# ═══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Ejemplo 7.8 — Teorema del Valor Medio para Integrales', fontsize=14, fontweight='bold')

# ── (a) CM(x) = 20 + 0.5x en [0,100] ────────────────────────
ax = axes[0]
CM_a = lambda x: 20 + 0.5*x
a_a, b_a = 0, 100
integral_a = quad(CM_a, a_a, b_a)[0]
f_bar_a = integral_a/(b_a-a_a)
c_a = (f_bar_a - 20)/0.5  # CM(c) = f_bar → 20+0.5c=45 → c=50

x_a = np.linspace(a_a, b_a, 400)
ax.plot(x_a, CM_a(x_a), color=C_AZUL, lw=2.5, label='CM(x)=20+0.5x')
ax.fill_between(x_a, 0, CM_a(x_a), alpha=0.2, color=C_AZUL, label=f'∫CM dx={integral_a:.0f}')
ax.axhline(f_bar_a, color=C_ROJO, lw=2, ls='--', label=f'CM_medio={f_bar_a:.1f}')
ax.axvline(c_a, color=C_GREEN if 'C_GREEN' in dir() else C_VERDE, lw=1.5, ls=':', label=f'c={c_a:.1f}')
ax.fill_between(x_a, f_bar_a, CM_a(x_a),
                where=CM_a(x_a)>=f_bar_a, alpha=0.2, color=C_VERDE, label='Sobre promedio')
ax.set_xlabel('x'); ax.set_ylabel('CM (miles S/u)')
ax.set_title(f'(a) CM=20+0.5x en [0,100]
Media={f_bar_a:.1f}, c={c_a:.1f}')
ax.legend(fontsize=8)
print(f"─── (a): Costo Medio Promedio ───")
print(f"  ∫₀¹⁰⁰ CM dx = {integral_a:.1f}")
print(f"  CM_medio = {f_bar_a:.1f}")
print(f"  CM(c)=media → 20+0.5c=45 → c={c_a:.1f}")

# ── (b) f(x)=x² en [0,3]: encontrar c ───────────────────────
ax = axes[1]
f_b = lambda x: x**2
a_b, b_b = 0, 3
integral_b = quad(f_b, a_b, b_b)[0]
f_bar_b = integral_b/(b_b-a_b)
c_b = np.sqrt(f_bar_b)  # f(c)=f_bar → c²=3 → c=√3

x_b = np.linspace(a_b, b_b, 400)
ax.plot(x_b, f_b(x_b), color=C_VERDE, lw=2.5, label='f(x)=x²')
ax.fill_between(x_b, 0, f_b(x_b), alpha=0.2, color=C_VERDE)
ax.axhline(f_bar_b, color=C_ROJO, lw=2, ls='--', label=f'Media={f_bar_b:.4f}=3')
ax.axvline(c_b, color=C_NARANJA, lw=1.5, ls=':', label=f'c=√3≈{c_b:.4f}')
ax.plot(c_b, f_bar_b, 'r*', ms=12)
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title(f'(b) f=x² en [0,3]
Media={f_bar_b:.2f}, c=√3={c_b:.4f}')
ax.legend(fontsize=9)
print(f"
─── (b): f(x)=x² en [0,3] ───")
print(f"  ∫₀³ x² dx = {integral_b:.4f}")
print(f"  Media = {integral_b}/{b_b-a_b} = {f_bar_b:.4f}")
print(f"  c = √{f_bar_b:.0f} = {c_b:.4f}  ✓")

# ── (c) Inflacion promedio con modelo periodico ───────────────
ax = axes[2]
pi_t = lambda t: 5 + 2*np.sin(np.pi*t/6)  # inflacion mensual ciclica
T_c = 12
integral_c = quad(pi_t, 0, T_c)[0]
pi_bar = integral_c/T_c

t_c = np.linspace(0, T_c, 400)
ax.plot(t_c, pi_t(t_c), color=C_MORADO, lw=2.5, label='π(t)=5+2sin(πt/6)')
ax.fill_between(t_c, 0, pi_t(t_c), alpha=0.2, color=C_MORADO)
ax.axhline(pi_bar, color=C_ROJO, lw=2, ls='--', label=f'Inflacion media={pi_bar:.2f}%')
ax.axhline(5, color='gray', lw=1, ls=':', label='Linea base 5%')
ax.set_xlabel('Mes'); ax.set_ylabel('Inflacion (%)')
ax.set_xticks(range(0,13))
ax.set_title(f'(c) Inflacion Periodica
Promedio={pi_bar:.2f}%')
ax.legend(fontsize=9)
print(f"
─── (c): Inflacion Mensual ───")
print(f"  π(t) = 5 + 2·sen(πt/6)")
print(f"  Inflacion promedio anual = {pi_bar:.4f}%")
for mes, lbl in [(3,'Marzo'), (6,'Junio'), (9,'Sep'), (12,'Dic')]:
    pi_m = pi_t(mes)
    print(f"  π({mes}) = {pi_m:.2f}%  [{lbl}]")

plt.tight_layout()
plt.savefig('cap07_fig08_valor_medio.png', bbox_inches='tight', dpi=110)
plt.show()
print("
Figura guardada: cap07_fig08_valor_medio.png")


---
## Resumen del Capítulo 7

| Concepto | Fórmula | Aplicación Económica |
|----------|---------|---------------------|
| **Suma de Riemann** | $\sum f(c_i)\Delta x$ | Aproximación de áreas |
| **TFC Parte II** | $\int_a^b f = F(b)-F(a)$ | Cálculo exacto de acumulaciones |
| **Área entre curvas** | $\int_a^b [f-g]\,dx$ | Bienestar económico |
| **Exc. del Consumidor** | $\int_0^{Q^*}[D(Q)-P^*]\,dQ$ | Ganancia de los compradores |
| **Exc. del Productor** | $\int_0^{Q^*}[P^*-O(Q)]\,dQ$ | Ganancia de los vendedores |
| **Valor Presente** | $\int_0^T R(t)e^{-rt}\,dt$ | Evaluación de proyectos |
| **Perpetuidad** | $D/r$ | Flujo infinito descontado |
| **Valor Medio** | $\frac{1}{b-a}\int_a^b f$ | Promedios continuos |

---
*Capítulo 7 completado — Repositorio: github.com/JeelCueva/Metodos_Cuantitativos*
